# Combine and process Cleanalyze results

Author: **Niels J. de Winter** (*n.j.de.winter@vu.nl*)<br>
Assistant Professor Vrije Universiteit Amsterdam

## Load packages

In [1]:
import os
# import matplotlib.pyplot as plt
# from matplotlib import cm
# import matplotlib.dates as mdates
# import numpy as np
import pandas as pd
# import re
# from sklearn.linear_model import LinearRegression
# from scipy.stats import t
# from collections import OrderedDict

## Read all cleanalyze data

In [2]:
folder_path = "TE_data_with_growth_rates"
csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')] # List all CSV files in the folder
dataframes = {}

for file in csv_files:
    file_path = os.path.join(folder_path, file)
    df = pd.read_csv(file_path)
    dataframes[file] = df

# Print the first few rows of each DataFrame
for file, df in dataframes.items():
    print(f"Data from {file}:")
    print(df.head(), "\n")  # Display the first few rows of each DataFrame
    print(f"Shape: {df.shape}\n")  # Print the shape of the DataFrame
    print(f"Columns: {df.columns.tolist()}\n")  # Print the column names
    print("-" * 40)  # Separator for clarity

Data from B255_peakid.csv:
      depth  proxy_filtered      time         Xpos         Ypos         Ca43  \
0  0.445779        0.013676  62.33211  30944.10620  80703.27530  6301.588000   
1  0.891558        0.000000  62.44125  30944.46978  80703.01737  5301.123838   
2  1.337336        0.019326  62.55040  30944.83337  80702.75945  4600.846556   
3  1.783115        0.037763  62.65955  30945.19695  80702.50152  5751.322804   
4  2.228894        0.013795  62.76871  30945.56053  80702.24359  6251.562891   

    23Na/43Ca  25Mg/43Ca  43Ca/43Ca  55Mn/43Ca  88Sr/43Ca  138Ba/43Ca  \
0  199.059248   1.253571       1000   0.306491   0.013676         0.0   
1  235.482235   3.406310       1000   0.347474   0.000000         0.0   
2  284.331679   1.771388       1000   0.380053   0.019326         0.0   
3  221.307763   3.115197       1000   0.348871   0.037763         0.0   
4  205.558351   1.264443       1000   0.242893   0.013795         0.0   

         peak                    timing_info         

## Load environmental data

In [7]:
# Load the environmental data
env_data_path = "min60_2022.csv"
env_df = pd.read_csv(env_data_path, sep=None, engine='python')
print(env_df.head())

# Remove spaces from column names
env_df.columns = env_df.columns.str.replace(' ', '', regex=True)

# Parse the date column (TM)
env_df['TM'] = pd.to_datetime(env_df['TM'], format='%Y%m%d%H%M%S', errors='coerce')

            TM           ET      T  T_std  T_N  T_max  T_min   T_flag   \
0  202201010100  44562.04167  6.562  0.105    6  6.677  6.438      1.0   
1  202201010200  44562.08333  6.676  0.059    6  6.805  6.646      1.0   
2  202201010300  44562.12500  6.982   0.05    6  7.031  6.906      1.0   
3  202201010400  44562.16667  7.008  0.073    6  7.183  6.991      1.0   
4  202201010500  44562.20833  7.141  0.093    6  7.244  7.024      1.0   

       S  S_std  S_N   S_max   S_min   S_flag   
0  28.145  0.288    6  28.435  27.795      1.0  
1  28.465  0.224    6  29.005  28.415      1.0  
2  29.395  0.121    6  29.505  29.225      1.0  
3  29.385  0.208    6  29.875  29.335      1.0  
4   29.67  0.305    6  30.065  29.315      1.0  


### Combine environmental data with TE data

In [13]:
# For all dataframes in 'dataframes', merge with env_df and add only T, T_std, S, S_std columns
for fname, df in dataframes.items():
    # Ensure timing_info is datetime
    df['timing_info'] = pd.to_datetime(df['timing_info'], errors='coerce')
    # Merge and select only required columns from env_df
    merged = pd.merge_asof(
        df.sort_values('timing_info'),
        env_df[['TM', 'T', 'T_std', 'S', 'S_std']].sort_values('TM'),
        left_on='timing_info',
        right_on='TM',
        direction='nearest'
    )
    # Add/overwrite columns in the original dataframe
    dataframes[fname]['T'] = merged['T_y'].values
    dataframes[fname]['T_std'] = merged['T_std_y'].values
    dataframes[fname]['S'] = merged['S_y'].values
    dataframes[fname]['S_std'] = merged['S_std_y'].values

    # Print the updated DataFrame head
    print(f"Updated data from {fname}:")
    print(dataframes[fname].head(), "\n")  # Display the first few rows of the updated DataFrame

    # Save the updated DataFrame in a new folder
    output_folder = "TE_data_with_GR_T_S"
    os.makedirs(output_folder, exist_ok=True)  # Create the folder if it doesn't exist
    output_file_path = os.path.join(output_folder, fname)
    dataframes[fname].to_csv(output_file_path, index=False)  # Save the updated DataFrame to CSV


Updated data from B255_peakid.csv:
      depth  proxy_filtered      time         Xpos         Ypos         Ca43  \
0  0.445779        0.013676  62.33211  30944.10620  80703.27530  6301.588000   
1  0.891558        0.000000  62.44125  30944.46978  80703.01737  5301.123838   
2  1.337336        0.019326  62.55040  30944.83337  80702.75945  4600.846556   
3  1.783115        0.037763  62.65955  30945.19695  80702.50152  5751.322804   
4  2.228894        0.013795  62.76871  30945.56053  80702.24359  6251.562891   

    23Na/43Ca  25Mg/43Ca  43Ca/43Ca  55Mn/43Ca  ...  \
0  199.059248   1.253571       1000   0.306491  ...   
1  235.482235   3.406310       1000   0.347474  ...   
2  284.331679   1.771388       1000   0.380053  ...   
3  221.307763   3.115197       1000   0.348871  ...   
4  205.558351   1.264443       1000   0.242893  ...   

            timing_info_centered  interval_growth_rate  \
0            2022-10-25 10:00:00             20.329381   
1  2022-10-25 09:30:00.601135243     